# Notebook 08: Pipeline Parallelism

## Why This Matters

When a model has more layers than fit on a single GPU (even with FSDP or TP), pipeline parallelism assigns different layers to different GPUs. GPT-3 has 96 layers -- pipeline parallelism lets 8 GPUs each hold 12 layers and process the model as a production pipeline. The catch: naive pipelining creates GPU idle time (the "pipeline bubble"). Understanding micro-batching and how to minimize bubble fraction is a core LLM infrastructure problem.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. The Pipeline Parallelism Idea

Pipeline parallelism (PP) partitions the model's **layers** across GPUs:
- GPU 0: layers 1-12 (embedding + first 11 transformer layers)
- GPU 1: layers 13-24
- GPU 2: layers 25-36
- GPU 3: layers 37-48 (last transformer layers + output head)

Data flows forward through the pipeline (GPU 0 -> 1 -> 2 -> 3) and backward in reverse.

**The bubble problem**: In the simplest implementation:
1. GPU 0 processes the full batch forward
2. Sends activations to GPU 1, which processes forward
3. ... after the full forward pass completes ...
4. GPU 3 starts backward
5. Sends gradients to GPU 2, etc.

During this, GPU 0 is idle waiting for the backward pass to reach it. This "bubble" can waste 50%+ of GPU time.

In [ ]:
# Simulate naive pipeline parallelism (with bubble)
def simulate_naive_pipeline(n_stages, n_gpus, forward_time=1.0, backward_time=2.0):
    """
    Returns a timeline showing when each GPU is active.
    forward_time: time for one stage forward pass (normalized)
    backward_time: time for one stage backward pass
    """
    # Timeline: list of (gpu_id, start, end, op_type)
    events = []
    
    # Forward pass: sequential
    t = 0
    fwd_end_times = []
    for stage in range(n_stages):
        events.append((stage, t, t + forward_time, 'forward'))
        t += forward_time
        fwd_end_times.append(t)
    
    # Backward pass: reverse sequential
    for stage in reversed(range(n_stages)):
        events.append((stage, t, t + backward_time, 'backward'))
        t += backward_time
    
    total_time = t
    active_time = n_stages * (forward_time + backward_time)
    bubble_fraction = 1.0 - active_time / (n_stages * total_time)
    
    return events, total_time, bubble_fraction

# Visualize
n_stages = 4
events_naive, total_naive, bubble_naive = simulate_naive_pipeline(n_stages, n_stages)

print(f"Naive pipeline ({n_stages} stages):")
print(f"  Total time: {total_naive:.1f}")
print(f"  Bubble fraction: {bubble_naive:.1%}")
print()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

def plot_pipeline(ax, events, n_stages, total_time, title):
    colors = {'forward': 'steelblue', 'backward': 'tomato', 'idle': 'lightgray'}
    for stage in range(n_stages):
        ax.barh(stage, total_time, left=0, color=colors['idle'], alpha=0.3)
    
    for (stage, start, end, op_type) in events:
        ax.barh(stage, end - start, left=start, color=colors[op_type], 
                alpha=0.8, edgecolor='white', linewidth=0.5)
        ax.text(start + (end-start)/2, stage, op_type[0].upper(),
               ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    
    ax.set_yticks(range(n_stages))
    ax.set_yticklabels([f'GPU {i}' for i in range(n_stages)])
    ax.set_xlabel('Time')
    ax.set_title(title)
    ax.set_xlim(0, total_time)
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='steelblue', label='Forward'),
        Patch(facecolor='tomato', label='Backward'),
        Patch(facecolor='lightgray', alpha=0.3, label='Idle (bubble)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')

plot_pipeline(axes[0], events_naive, n_stages, total_naive,
              f'Naive Pipeline ({n_stages} stages) -- Bubble: {bubble_naive:.0%}')

# Show idle time per GPU
idle_per_gpu = []
for stage in range(n_stages):
    active = sum((e - s) for (g, s, e, t) in events_naive if g == stage)
    idle_per_gpu.append(total_naive - active)
    
axes[1].bar(range(n_stages), idle_per_gpu, color='lightcoral')
axes[1].set_xlabel('GPU Stage')
axes[1].set_ylabel('Idle time')
axes[1].set_title('Idle time per GPU in naive pipeline')
axes[1].set_xticks(range(n_stages))
axes[1].set_xticklabels([f'GPU {i}' for i in range(n_stages)])

plt.tight_layout()
plt.savefig('naive_pipeline.png', dpi=100)
plt.show()

## 2. Micro-Batching: The GPipe Approach

The solution to the pipeline bubble: split each mini-batch into M **micro-batches** and pipeline them.

With M micro-batches and N pipeline stages:
- GPU 0 processes micro-batch 1 forward, sends to GPU 1
- While GPU 1 processes micro-batch 1, GPU 0 processes micro-batch 2
- ...

The bubble fraction is:
$$\text{bubble fraction} = \frac{N-1}{M + N - 1}$$

For M >> N: bubble fraction → 0. With M=8 micro-batches and N=4 stages: bubble = 3/(8+4-1) = 27%.

**The tradeoff**: more micro-batches = smaller bubble, but each micro-batch must have enough data for good GPU utilization. Very small micro-batches are memory-bandwidth-bound.

In [ ]:
# GPipe-style micro-batch pipeline simulation

def simulate_gpipe(n_stages, n_micro_batches, forward_time=1.0, backward_time=2.0):
    """
    Simulate GPipe: all-forward then all-backward (F^M B^M schedule).
    Bubble fraction = (N-1) / (M + N - 1)
    """
    events = []
    
    # All forwards first
    stage_ready = [0.0] * n_stages  # when each stage is next available
    mb_done_fwd = [0.0] * n_micro_batches  # when each mb finishes forward at last stage
    
    for mb in range(n_micro_batches):
        t = 0
        for stage in range(n_stages):
            start = max(stage_ready[stage], t)
            end = start + forward_time
            events.append((stage, start, end, 'forward', mb))
            stage_ready[stage] = end
            t = end
        mb_done_fwd[mb] = t
    
    # All backwards (reverse mb order)
    stage_ready_bwd = [mb_done_fwd[-1]] * n_stages
    for mb in reversed(range(n_micro_batches)):
        t = mb_done_fwd[mb]
        for stage in reversed(range(n_stages)):
            start = max(stage_ready_bwd[stage], t)
            end = start + backward_time
            events.append((stage, start, end, 'backward', mb))
            stage_ready_bwd[stage] = end
            t = end
    
    total_time = max(e for (_, _, e, _, _) in events)
    active_per_stage = [(forward_time + backward_time) * n_micro_batches] * n_stages
    bubble = 1 - active_per_stage[0] / total_time
    
    theoretical_bubble = (n_stages - 1) / (n_micro_batches + n_stages - 1)
    return events, total_time, bubble, theoretical_bubble

n_stages = 4
fig, axes = plt.subplots(len([2, 4, 8]), 1, figsize=(14, 12))

for ax, n_mb in zip(axes, [2, 4, 8]):
    events, total, bubble, theoretical = simulate_gpipe(n_stages, n_mb)
    
    colors_mb = plt.cm.Paired(np.linspace(0, 1, n_mb))
    
    for stage in range(n_stages):
        ax.barh(stage, total, left=0, color='lightgray', alpha=0.3)
    
    for (stage, start, end, op_type, mb) in events:
        color = colors_mb[mb]
        alpha = 0.8 if op_type == 'forward' else 0.5
        ax.barh(stage, end - start, left=start, color=color, alpha=alpha,
               edgecolor='white', linewidth=0.5)
    
    ax.set_yticks(range(n_stages))
    ax.set_yticklabels([f'GPU {i}' for i in range(n_stages)])
    ax.set_title(f'GPipe: {n_stages} stages, {n_mb} micro-batches | '
                 f'Bubble: {bubble:.1%} (theory: {theoretical:.1%})')

plt.tight_layout()
plt.savefig('gpipe_pipeline.png', dpi=100)
plt.show()

print("Bubble fraction formula: (N-1)/(M+N-1)")
print()
print(f"{'Micro-batches (M)':<22} {'Bubble (N=4)'}")
print("-" * 35)
for m in [1, 2, 4, 8, 16, 32, 64]:
    n = 4
    bubble = (n - 1) / (m + n - 1)
    print(f"{m:<22} {bubble:.1%}")

## 3. 1F1B Schedule: Interleaved Forward and Backward

PipeDream and Megatron-LM's 1F1B (one forward, one backward) schedule reduces peak activation memory while keeping GPU utilization similar to GPipe.

The 1F1B steady state:
- Each GPU alternates: run one forward micro-batch, then one backward micro-batch
- This keeps the pipeline full while reducing the number of in-flight activations

**Activation memory comparison**:
- GPipe: all M forward passes complete before any backward -> O(M * N) activations in memory
- 1F1B: at steady state, only O(N) micro-batches worth of activations active at once

This is the schedule used in Megatron-LM v2 and later, and it's the reason large transformers are trainable with pipeline parallelism without OOM on activations.

In [ ]:
# 1F1B schedule simulation

def simulate_1f1b(n_stages, n_micro_batches, forward_time=1.0, backward_time=2.0):
    """Simulate the 1F1B interleaved schedule."""
    events = []
    
    # For simplicity, simulate a clock-based schedule
    # Stage i starts forward of mb j when mb j has finished stage i-1
    
    fwd_done = {}  # (mb, stage) -> end_time
    bwd_done = {}  # (mb, stage) -> end_time
    stage_fwd_avail = [0.0] * n_stages
    stage_bwd_avail = [0.0] * n_stages
    
    # Startup: fill the pipeline with forwards
    for mb in range(n_stages):
        for stage in range(mb + 1):
            prev_done = fwd_done.get((mb, stage - 1), 0.0) if stage > 0 else 0.0
            start = max(stage_fwd_avail[stage], prev_done)
            end = start + forward_time
            fwd_done[(mb, stage)] = end
            stage_fwd_avail[stage] = end
            events.append((stage, start, end, 'forward', mb))
    
    # Steady state: 1F1B
    # Forward then backward for each mb in order
    for mb in range(n_stages, n_micro_batches):
        # Forward pass for mb
        for stage in range(n_stages):
            prev_done = fwd_done.get((mb, stage - 1), 0.0) if stage > 0 else 0.0
            start = max(stage_fwd_avail[stage], prev_done)
            end = start + forward_time
            fwd_done[(mb, stage)] = end
            stage_fwd_avail[stage] = end
            events.append((stage, start, end, 'forward', mb))
    
    # Backward passes
    for mb in range(n_micro_batches):
        for stage in reversed(range(n_stages)):
            fwd_complete = fwd_done[(mb, n_stages - 1)]
            prev_bwd = bwd_done.get((mb, stage + 1), fwd_complete) if stage < n_stages - 1 else fwd_complete
            start = max(stage_bwd_avail[stage], prev_bwd)
            end = start + backward_time
            bwd_done[(mb, stage)] = end
            stage_bwd_avail[stage] = end
            events.append((stage, start, end, 'backward', mb))
    
    total_time = max(e for (_, _, e, _, _) in events)
    active_per_stage = (forward_time + backward_time) * n_micro_batches
    bubble = 1 - active_per_stage / total_time
    return events, total_time, bubble

# Compare GPipe vs 1F1B
n_stages, n_mb = 4, 8
_, t_gpipe, b_gpipe, _ = simulate_gpipe(n_stages, n_mb)
_, t_1f1b, b_1f1b = simulate_1f1b(n_stages, n_mb)

print(f"Comparison: {n_stages} stages, {n_mb} micro-batches")
print(f"{'Schedule':<15} {'Total time':<15} {'Bubble':<15} {'Peak activations'}")
print("-" * 60)
print(f"{'GPipe':<15} {t_gpipe:<15.1f} {b_gpipe:<15.1%} {'O(M*N)=' + str(n_mb*n_stages)}")
print(f"{'1F1B':<15} {t_1f1b:<15.1f} {b_1f1b:<15.1%} {'O(N)=' + str(n_stages)}")
print()
print(f"1F1B saves {n_mb}x activation memory vs GPipe!")
print(f"(Bubble is similar, but memory footprint is drastically lower)")

## 4. Inter-Stage Communication and Bandwidth

Each pipeline stage boundary requires sending **activations** forward and **gradients** backward.

For a transformer with hidden size `d_model=8192` and sequence length `seq_len=2048`, batch `B=8`:
- Activation tensor: `B * seq_len * d_model * 2 bytes` = 8 * 2048 * 8192 * 2 = 268 MB
- This must cross PCIe (CPU-GPU, ~16 GB/s) or NVLink (600 GB/s) or InfiniBand (200 GB/s)

**Critical design decision**: pipeline stages should be on GPUs with the highest inter-GPU bandwidth. Within a node (NVLink): fast. Across nodes (InfiniBand): must be carefully managed.

In [ ]:
# Inter-stage communication analysis

def pipeline_comm_time_ms(d_model, seq_len, batch, dtype_bytes, bandwidth_gbs):
    activation_bytes = batch * seq_len * d_model * dtype_bytes
    return activation_bytes / (bandwidth_gbs * 1e9) * 1000

configs = [
    ("NVLink (600 GB/s)", 600),
    ("InfiniBand HDR (200 GB/s)", 200),
    ("PCIe Gen4 (32 GB/s)", 32),
]

d_model, seq_len, batch = 8192, 2048, 8
print(f"Activation transfer time (d_model={d_model}, seq={seq_len}, batch={batch}, bf16):")
print()
for name, bw in configs:
    t = pipeline_comm_time_ms(d_model, seq_len, batch, 2, bw)
    print(f"  {name:<30}: {t:.1f} ms")

print()
# Estimate compute time per stage for 12 transformer layers
# ~2x flops of forward = 2 * 2 * B * seq * d_model^2 * (layers_per_stage) 
layers_per_stage = 12
flops_per_stage = 2 * 2 * batch * seq_len * d_model**2 * layers_per_stage
tflops = 312  # A100 theoretical
compute_ms = flops_per_stage / (tflops * 1e12) * 1000

print(f"Compute time per stage ({layers_per_stage} layers): {compute_ms:.1f} ms")
print()
print("Communication/compute ratio:")
for name, bw in configs:
    t = pipeline_comm_time_ms(d_model, seq_len, batch, 2, bw)
    ratio = t / compute_ms
    bottleneck = "COMM-BOUND" if ratio > 0.1 else "OK"
    print(f"  {name:<30}: ratio={ratio:.3f} ({bottleneck})")

## Summary

### Key Concepts

| Schedule | Bubble fraction | Peak activations | Used in |
|----------|----------------|-----------------|---------|
| Naive (no MB) | (N-1)/N | 1 MB | Never in practice |
| GPipe | (N-1)/(M+N-1) | M*N micro-batches | Original GPipe paper |
| 1F1B | (N-1)/(M+N-1) | N micro-batches | Megatron-LM v2+ |
| Interleaved 1F1B | (N-1)/(M*V+N-1) | N micro-batches | Megatron-LM v3 |

**Choosing M (micro-batches)**:
- Target: bubble fraction < 5% -> M >= 20*N roughly
- Minimum micro-batch size: enough data to keep GPU compute-bound
- Typical: M=8-32 for N=4-8 stage pipelines

**3D parallelism** (Megatron-LM): Tensor Parallelism (within node) × Pipeline Parallelism (across nodes) × Data Parallelism (across replicas). This is how 530B+ parameter models are trained.

**Next**: `09_megatron_lm_patterns.ipynb` -- how all three parallelism strategies combine in real LLM training.